In [3]:
import ollama
import json
import re

# -----------------------------
# 1. CALL LLM
# -----------------------------
def call_llm(user_input):
    prompt = f"""
You are a system dynamics expert. Your job is to extract stock-and-flow structures from text descriptions.

Text:
\"\"\"{user_input}\"\"\"

REQUIRED OUTPUT FORMAT - You MUST return valid JSON with ALL of these keys:
{{
  "stocks": [],
  "inflows": [],
  "outflows": [],
  "auxiliary": [],
  "equations": {{}},
  "units": {{}}
}}

RULES:

1. STOCKS accumulate over time (e.g., Population, Water Level)
2. INFLOWS increase stocks (unit = stock/time)
3. OUTFLOWS decrease stocks (unit = stock/time)
4. AUXILIARY variables:
   - constants
   - ratios
   - probabilities
   - delays / time constants
   - intermediate calculations

5. DELAY HANDLING (STRICT RULE):
   If the text mentions "delay", "lag", "time to respond", "time to adjust":
   → MUST create an auxiliary variable representing that delay
   → Use it inside equations

6. EQUATIONS:
   - Each stock MUST use INTEG(inflow - outflow, initial_value)
   - Use SIMPLE linear relationships unless clearly specified
   - Avoid complex nonlinear formulas unless necessary

7. UNITS (VERY IMPORTANT):
   - Every variable MUST have a unit
   - Stocks: base unit (e.g., people, houses, dollars)
   - Flows: stock/time (e.g., people/month)
   - Delays: time (e.g., month)
   - Probabilities: dimensionless
   - Rates: 1/time

8. NAMING:
   - Use clear, readable names (no abbreviations)
   - Be consistent across equations and units

9. OUTPUT RULE:
   Return ONLY JSON. No explanation. No markdown.



EXAMPLE:

Input: "Population increases through births and decreases through deaths. There is a delay in response to overcrowding."

Output:
{{
  "stocks": ["Population"],
  "inflows": ["Births"],
  "outflows": ["Deaths"],
  "auxiliary": ["Birth Rate", "Death Rate", "Response Delay"],
  "equations": {{
    "Population": "INTEG(Births - Deaths, 1000)",
    "Births": "Population * Birth Rate",
    "Deaths": "Population * Death Rate",
    "Birth Rate": "0.02",
    "Death Rate": "0.01",
    "Response Delay": "5"
  }},
  "units": {{
    "Population": "people",
    "Births": "people/month",
    "Deaths": "people/month",
    "Birth Rate": "1/month",
    "Death Rate": "1/month",
    "Response Delay": "month"
  }}
}}

Now analyze the given text and return JSON only.
"""
    try:
        response = ollama.chat(
            model='qwen3.6',
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

    except Exception as e:
        print("LLM call failed:", e)
        return ""



# -----------------------------
# 2. PARSE JSON SAFELY
# -----------------------------
def parse_json(response_text):
    try:
        # Try direct parsing first
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract JSON from messy output (handles markdown code blocks)
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(1))
    except json.JSONDecodeError:
        pass
    
    try:
        # Extract raw JSON object
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except json.JSONDecodeError as e:
        print("JSON parsing error:", e)
    
    return None

# -----------------------------
# 2B. EXTRACT MISSING FIELDS FROM JSON
# -----------------------------
def extract_missing_fields(data):
    """If stocks/inflows/outflows are missing, try to extract from equations"""
    if not data or not data.get("equations"):
        return data
    
    equations = data["equations"]
    
    # Extract stocks (those with INTEG)
    if not data.get("stocks"):
        stocks = [var for var, eq in equations.items() if "INTEG" in eq.upper()]
        if stocks:
            data["stocks"] = stocks
    
    # Extract inflows (mentioned in INTEG right side before minus)
    if not data.get("inflows"):
        inflows = set()
        for var, eq in equations.items():
            if "INTEG" in eq.upper():
                # Parse "INTEG(inflow - outflow, init)"
                match = re.search(r'INTEG\s*\(\s*([^-]+?)\s*-', eq, re.IGNORECASE)
                if match:
                    inflows.add(match.group(1).strip())
        if inflows:
            data["inflows"] = list(inflows)
    
    # Extract outflows (mentioned in INTEG right side after minus)
    if not data.get("outflows"):
        outflows = set()
        for var, eq in equations.items():
            if "INTEG" in eq.upper():
                # Parse "INTEG(inflow - outflow, init)"
                match = re.search(r'INTEG\s*\([^-]*-\s*([^,]+?)\s*,', eq, re.IGNORECASE)
                if match:
                    outflows.add(match.group(1).strip())
        if outflows:
            data["outflows"] = list(outflows)
    
    return data
 

# -----------------------------
# 3. VALIDATION
# -----------------------------
def validate(data):
    errors = []
 
    if not data:
        errors.append("No data parsed")
        return errors
 
    # Check for required keys (using plural forms to match prompt)
    if not data.get("stocks") or len(data.get("stocks", [])) == 0:
        errors.append("Missing stocks")
 
    if not data.get("inflows") or len(data.get("inflows", [])) == 0:
        errors.append("No inflows defined")
 
    if not data.get("outflows") or len(data.get("outflows", [])) == 0:
        errors.append("No outflows defined")
 
    if not data.get("equations") or len(data.get("equations", {})) == 0:
        errors.append("No equations provided")

    return errors


# -----------------------------
# 4. GENERATE MDL
# -----------------------------
def generate_mdl(data):
    lines = []

    # Header
    lines.append("{UTF-8}")

    equations = data.get("equations", {})
    units = data.get("units", {})

    def format_var(name):
        return name.lower()

    def format_unit(unit):
        """Convert to Vensim style (Month instead of month)"""
        if not unit:
            return "dimensionless"
        return unit.replace("month", "Month").replace("per", "/")


    # Generate variables
    for var, eq in equations.items():
        var_name = format_var(var)
        eq_formatted = eq.strip()

        unit = format_unit(units.get(var, "dimensionless"))

        lines.append(f"{var_name} = {eq_formatted}")
        lines.append(f"~ {unit}")
        lines.append(f"~ |")
        lines.append("")

    # Control section
    lines.append("********************************************************")
    lines.append(".Control")
    lines.append("********************************************************~")
    lines.append("Simulation Control Parameters")
    lines.append("|")
    lines.append("")
    lines.append("FINAL TIME  = 100")
    lines.append("~ Month")
    lines.append("~ The final time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("INITIAL TIME  = 0")
    lines.append("~ Month")
    lines.append("~ The initial time for the simulation.")
    lines.append("|")
    lines.append("")
    lines.append("SAVEPER  =")
    lines.append("        TIME STEP")
    lines.append("~ Month")
    lines.append("~ The frequency with which output is stored.")
    lines.append("|")
    lines.append("")
    lines.append("TIME STEP  = 1")
    lines.append("~ Month")
    lines.append("~ The time step for the simulation.")
    lines.append("|")

    return "\n".join(lines)


# -----------------------------
# 5. FULL PIPELINE
# -----------------------------
def run_pipeline(user_input):
    print("🔹 Input:", user_input)
    print("\n" + "="*70)
 
    raw = call_llm(user_input)
    print("\n🔹 Raw LLM Output:")
    print(raw)
    print("="*70)
 
    parsed = parse_json(raw)
    print("\n🔹 Parsed JSON (before extraction):")
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print("Failed to parse JSON")
    
    # Try to extract missing stocks/inflows/outflows from equations
    parsed = extract_missing_fields(parsed)
    print("\n🔹 After field extraction:")
    if parsed:
        print(json.dumps(parsed, indent=2))
    print("="*70)
 
    errors = validate(parsed)
 
    if errors:
        print("\n❌ Validation Errors:")
        for error in errors:
            print(f"  - {error}")
        return None
 
    mdl = generate_mdl(parsed)
    print("\n✅ MDL Output:")
    print(mdl)
    print("="*70)
 
    return parsed, mdl

In [2]:
run_pipeline("More infected people increase infection rate, recovery reduces infected")

NameError: name 'run_pipeline' is not defined